### dividend.parquet数据

In [3]:
from warnings import filterwarnings
filterwarnings('ignore')

In [4]:
import pandas as pd
dividend = pd.read_parquet('data/dividend.parquet')
dividend.head()

,stock_code,announce_date,ex_date,div_pre_tax,div_after_tax,progress,base_shares,div_year,currency,is_no_div,total_cash_div
0,002440.SZ,20131009,20131015,0.000000,0.000000,3,38346.810526,20130630,CNY,0,NaN
1,603339.SH,20180717,20180723,0.180003,0.179996,3,21086.028397,20171231,CNY,0,3.795272e+07
2,002682.SZ,20150515,20150522,0.070010,0.066502,3,20797.888645,20141231,CNY,0,1.456142e+07
3,002740.SZ,20160401,20160411,0.109993,0.109997,3,9999.096297,20151231,CNY,0,1.100072e+07
4,002840.SZ,20170627,20170705,0.104990,0.105007,3,17868.104914,20161231,CNY,0,1.875843e+07


In [2]:
dividend = dividend.sort_values(by=['stock_code', 'ex_date'], ascending=[True, True])
dividend.head()


,stock_code,announce_date,ex_date,div_pre_tax,div_after_tax,progress,base_shares,div_year,currency,is_no_div,total_cash_div
8165,000001.SZ,20130614,20130620,0.170001,0.131472,3,5.123809e+05,20121231,CNY,0,8.709306e+08
6411,000001.SZ,20140606,20140612,0.159974,0.151981,3,9.521493e+05,20131231,CNY,0,1.523397e+09
20151,000001.SZ,20150407,20150413,0.173981,0.165298,3,1.142486e+06,20141231,CNY,0,1.988210e+09
21055,000001.SZ,20160608,20160616,0.152994,0.152997,3,1.430813e+06,20151231,CNY,0,2.189083e+09
3884,000001.SZ,20170717,20170721,0.157993,0.158005,3,1.717085e+06,20161231,CNY,0,2.712675e+09


初级：*每股税后股息(div_after_tax)*
进阶：*股息率 = 每股税后股息(div_after_tax) / 除权除息日股价(ex_date)*
**考虑某些年度不分红的情况,is_no_div字段**
绝对避免未来函数：选股时只能使用调仓日已公开且生效的股息率数据；
以除权日为准：股息率的生效时间是除权除息日，而非公告日——只有除权后，股息才真正兑现，股价也完成除权，此时股息率才具备实际选股意义；
调仓频率适配股息更新节奏：股息率是年度维度的因子，月度 / 季度调仓即可；
因子时效性：用最新一期已生效的股息率。

In [ ]:
### eod_prices.parquet数据

In [ ]:
price_dt = pd.read_parquet('data/eod_prices.parquet')
price_dt=price_dt.sort_values(by=['stock_code', 'trade_date'], ascending=[True, True])
price_dt.head()


,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover
2983,000001.SZ,20130104,15.99,355.0546,22.204789,443851.37,717567.5466
8347,000001.SZ,20130107,16.30,361.9381,22.204789,357169.25,578450.4876
10530,000001.SZ,20130108,16.00,355.2766,22.204789,312479.12,501360.0937
13553,000001.SZ,20130109,15.86,352.1680,22.204789,251329.15,399696.1831
10100,000001.SZ,20130110,15.87,352.3900,22.204789,240030.27,383347.7326


close和adj_close，回测时候要用复权adj_close

### monthly_data.parquet数据

In [12]:
capital_dt = pd.read_parquet('data/capital.parquet')
capital_dt=capital_dt.sort_values(by=['stock_code', 'change_date'], ascending=[True, True])
capital_dt.head()


,stock_code,change_date,total_shares,float_shares,total_a_shares,float_a_shares
4211,000001.SZ,20130614,512437.850182,310502.606910,512363.631057,310533.405247
3199,000001.SZ,20130620,819808.237690,496861.283537,819958.402672,496892.496412
3081,000001.SZ,20131112,819787.022887,557587.137080,819660.685162,557526.336922
2748,000001.SZ,20140109,952245.621468,557576.499569,952199.483331,557628.702797
5321,000001.SZ,20140606,952167.083977,557652.723241,952107.503712,557532.647011


*流通市值 = close_price * float_shares*
*总市值 = close_price * total_shares*
*市值因子 = -Ln(流通市值) 对于行业做标准化*

### 个股行业分类用哪个

In [19]:
stk_cat=pd.read_parquet('data/ref_data/AShareSWNIndustriesClass.parquet')
stk_cat.sort_values(by=['S_INFO_WINDCODE'], ascending=[True])
stk_cat.head()


,S_INFO_WINDCODE,SW_IND_CODE,ENTRY_DT,REMOVE_DT,CUR_SIGN,INDUSTRIESNAME
0,600819.SH,760k020100,1994-01-28,NaT,1,建筑材料
1,600820.SH,760l030100,1994-01-28,NaT,1,建筑装饰
2,600821.SH,760f020100,1994-01-28,2022-07-28,0,商贸零售
3,600822.SH,760f010100,1994-02-04,2021-07-29,0,商贸零售
4,002801.SZ,760m010100,2016-06-22,NaT,1,电力设备


In [21]:
df=pd.read_parquet('data/ref_data/Stock_Industry_Year.parquet')
df.head()


,stock_code,year,industry_name
0,600819.SH,2010,建筑材料
1,600820.SH,2010,建筑装饰
2,600821.SH,2010,商贸零售
3,600822.SH,2010,商贸零售
4,600825.SH,2010,传媒


### 辅助股票池：ETF持仓

In [22]:
df=pd.read_parquet('data/ref_data/ETF_hold_510300.SH.parquet')
df.head()

,fund_code,end_date,stock_code,index_value,report_year,report_type
0,510300.SH,2025-06-30,300413.SZ,2.935375e+08,2025,中报
1,510300.SH,2025-06-30,600018.SH,2.860284e+08,2025,中报
2,510300.SH,2025-06-30,601607.SH,2.695809e+08,2025,中报
3,510300.SH,2025-06-30,600026.SH,2.578523e+08,2025,中报
4,510300.SH,2025-06-30,688303.SH,2.487415e+08,2025,中报


In [23]:
df=pd.read_parquet('data/ref_data/ETF_hold_510500.SH.parquet')
df.head()

,fund_code,end_date,stock_code,index_value,report_year,report_type
0,510500.SH,2025-06-30,001213.SZ,53516484.00,2025,中报
1,510500.SH,2025-06-30,601921.SH,52849420.00,2025,中报
2,510500.SH,2025-06-30,601858.SH,44948677.70,2025,中报
3,510500.SH,2025-06-30,688153.SH,40143654.39,2025,中报
4,510500.SH,2025-06-30,601158.SH,40117904.00,2025,中报


In [26]:
df=pd.read_parquet('data/ref_data/ETF_hold_512100.SH.parquet')
df.head()

,fund_code,end_date,stock_code,index_value,report_year,report_type
0,512100.SH,2025-06-30,300660.SZ,74000248.00,2025,中报
1,512100.SH,2025-06-30,300352.SZ,73918957.00,2025,中报
2,512100.SH,2025-06-30,000762.SZ,73785460.00,2025,中报
3,512100.SH,2025-06-30,300687.SZ,73572552.00,2025,中报
4,512100.SH,2025-06-30,603360.SH,73501122.66,2025,中报


510300.SH	华泰柏瑞沪深 300ETF
510500.SH	南方中证 500ETF
512100.SH	南方中证 1000ETF